1. Pasang paket

In [1]:
%pip install -U "transformers==5.22.1" accelerate peft "bitsandbytes>=0.46.1" datasets pillow

Kita menggunakan versi transformers yang sebelumnya berhasil di notebookmu.

Setelah selesai, pilih Runtime → Restart session. Lanjutkan dari langkah 2, tanpa mengulang langkah 1.

2. Periksa perangkat dan paket

In [2]:
import torch
import transformers
import bitsandbytes
import peft

print("Transformers:", transformers.__version__)
print("Bitsandbytes:", bitsandbytes.__version__)
print("GPU tersedia:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU belum aktif. Pilih Runtime → Change runtime type → GPU."
    )

gpu = torch.cuda.get_device_properties(0)

print("Nama GPU:", gpu.name)
print("Memori GPU:", round(gpu.total_memory / 1024**3, 1), "GiB")

Transformers: 5.22.1
Bitsandbytes: 0.50.2
GPU tersedia: True
Nama GPU: Tesla T4
Memori GPU: 14.6 GiB


Jenis GPU bisa berbeda dari sesi sebelumnya. Jika bukan T4 atau memorinya lebih kecil, perhatikan kapasitasnya sebelum melanjutkan.

3. Hubungkan Google Drive

In [3]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Ikuti permintaan izin akses Drive yang muncul.


4. Periksa folder model

In [4]:
from pathlib import Path

lokasi_model = Path(
    "/content/drive/MyDrive/LeveLearn/models/Qwen3.5-9B"
)

if not (lokasi_model / "config.json").exists():
    raise FileNotFoundError(
        "Folder model belum ditemukan. Periksa lokasi folder di Google Drive."
    )

print("Folder model ditemukan.")
print("Jumlah berkas bobot:", len(list(lokasi_model.glob("*.safetensors"))))

Folder model ditemukan.
Jumlah berkas bobot: 4


5. Muat model ke GPU
Jalankan sekali saja pada sesi baru. Jika model sudah berhasil dimuat, tidak perlu mengulang sel ini.

In [5]:
from transformers import (
    AutoProcessor,
    BitsAndBytesConfig,
    Qwen3_5ForConditionalGeneration,
)

pengaturan_memori = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

processor = AutoProcessor.from_pretrained(
    str(lokasi_model),
    local_files_only=True,
)

model = Qwen3_5ForConditionalGeneration.from_pretrained(
    str(lokasi_model),
    local_files_only=True,
    quantization_config=pengaturan_memori,
    device_map={"": 0},
    dtype=torch.float16,
    attn_implementation="sdpa",
)

model.eval()

print("Model berhasil dimuat.")
print(
    "Memori GPU terpakai:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GiB",
)

Loading weights:   0%|          | 0/760 [00:00<?, ?it/s]

Model berhasil dimuat.
Memori GPU terpakai: 7.35 GiB


6. Siapkan fungsi untuk bertanya
Dengan fungsi ini, kita tidak perlu menulis ulang seluruh kode pembangkitan setiap kali bertanya.

In [6]:
def tanya_model(pertanyaan):
    pesan = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": pertanyaan}
            ],
        }
    ]

    masukan = processor.apply_chat_template(
        pesan,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_dict=True,
        return_tensors="pt",
    ).to("cuda")

    with torch.inference_mode():
        keluaran = model.generate(
            **masukan,
            max_new_tokens=150,
            do_sample=False,
        )

    token_jawaban = keluaran[:, masukan["input_ids"].shape[1]:]

    jawaban = processor.batch_decode(
        token_jawaban,
        skip_special_tokens=True,
    )[0]

    del masukan, keluaran, token_jawaban
    torch.cuda.empty_cache()

    return jawaban


print("Fungsi untuk bertanya sudah siap.")

Fungsi untuk bertanya sudah siap.


Lalu coba:

In [7]:
pertanyaan_tes = (
    "Jawab dalam bahasa Indonesia dengan singkat. "
    "Jika p bernilai benar dan q bernilai salah, "
    "berapa nilai kebenaran p ∧ q? Jelaskan alasannya."
)

jawaban_sebelum = tanya_model(pertanyaan_tes)

print(jawaban_sebelum)

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


Nilai kebenaran **Salah**.

**Alasan:**
Dalam operasi logika konjungsi (∧), pernyataan bernilai benar hanya jika **kedua** pernyataan penyusunnya bernilai benar. Karena dalam kasus ini p benar tetapi q salah, maka hasilnya adalah salah.



7. Siapkan kembali enam contoh latihan

In [8]:
data_latihan = [
    {
        "pertanyaan": "Jika p benar dan q benar, tentukan nilai p ∧ q.",
        "jawaban": (
            "Hasil: Benar.\n"
            "Aturan: Konjungsi benar hanya jika kedua pernyataan benar.\n"
            "Penerapan: p benar dan q benar, sehingga p ∧ q benar."
        ),
    },
    {
        "pertanyaan": "Jika p benar dan q salah, tentukan nilai p ∧ q.",
        "jawaban": (
            "Hasil: Salah.\n"
            "Aturan: Konjungsi benar hanya jika kedua pernyataan benar.\n"
            "Penerapan: q salah, sehingga p ∧ q salah."
        ),
    },
    {
        "pertanyaan": "Jika p salah dan q benar, tentukan nilai p ∨ q.",
        "jawaban": (
            "Hasil: Benar.\n"
            "Aturan: Disjungsi benar jika sedikitnya satu pernyataan benar.\n"
            "Penerapan: q benar, sehingga p ∨ q benar."
        ),
    },
    {
        "pertanyaan": "Jika p salah dan q salah, tentukan nilai p ∨ q.",
        "jawaban": (
            "Hasil: Salah.\n"
            "Aturan: Disjungsi salah jika kedua pernyataan salah.\n"
            "Penerapan: p salah dan q salah, sehingga p ∨ q salah."
        ),
    },
    {
        "pertanyaan": "Jika p benar, tentukan nilai ¬p.",
        "jawaban": (
            "Hasil: Salah.\n"
            "Aturan: Negasi membalik nilai kebenaran pernyataan.\n"
            "Penerapan: p benar, sehingga ¬p salah."
        ),
    },
    {
        "pertanyaan": "Jika p salah, tentukan nilai ¬p.",
        "jawaban": (
            "Hasil: Benar.\n"
            "Aturan: Negasi membalik nilai kebenaran pernyataan.\n"
            "Penerapan: p salah, sehingga ¬p benar."
        ),
    },
]

print("Jumlah contoh latihan:", len(data_latihan))

Jumlah contoh latihan: 6


In [9]:
from datasets import Dataset

tokenizer = processor.tokenizer

def siapkan_contoh(contoh):
    pesan = [
        {
            "role": "user",
            "content": contoh["pertanyaan"],
        }
    ]

    # Format pertanyaan, sama seperti ketika model digunakan.
    awalan = tokenizer.apply_chat_template(
        pesan,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    token_pertanyaan = tokenizer(
        awalan,
        add_special_tokens=False,
    )["input_ids"]

    token_jawaban = tokenizer(
        contoh["jawaban"],
        add_special_tokens=False,
    )["input_ids"] + [tokenizer.eos_token_id]

    input_ids = token_pertanyaan + token_jawaban

    # -100 berarti bagian pertanyaan tidak diberi nilai kesalahan.
    labels = [-100] * len(token_pertanyaan) + token_jawaban

    if len(input_ids) > 256:
        raise ValueError("Contoh terlalu panjang untuk pengaturan latihan ini.")

    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": labels,
    }

dataset_latihan = Dataset.from_list(
    [siapkan_contoh(contoh) for contoh in data_latihan]
)

print("Jumlah contoh siap latihan:", len(dataset_latihan))

Jumlah contoh siap latihan: 6


In [10]:
import torch
from peft import LoraConfig, get_peft_model

if hasattr(model, "peft_config"):
    raise RuntimeError(
        "LoRA sudah terpasang. Jangan ulangi sel ini."
    )

# Bobot utama dibekukan, tanpa mengubah seluruhnya ke 32-bit.
for parameter in model.parameters():
    parameter.requires_grad_(False)

# Hanya parameter kecil untuk normalisasi yang dibuat 32-bit.
for nama, parameter in model.named_parameters():
    if "norm" in nama.lower() and parameter.ndim == 1:
        parameter.data = parameter.data.to(torch.float32)

model.config.use_cache = False

if hasattr(model.config, "text_config"):
    model.config.text_config.use_cache = False

# Mengurangi penyimpanan data sementara saat pelatihan.
model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)

modul_sasaran = [
    nama
    for nama, modul in model.named_modules()
    if ".self_attn." in nama
    and nama.endswith((".q_proj", ".v_proj"))
    and hasattr(modul, "weight")
]

if not modul_sasaran:
    raise RuntimeError("Bagian sasaran tidak ditemukan.")

pengaturan_lora = LoraConfig(
    r=4,
    lora_alpha=8,
    lora_dropout=0.0,
    target_modules=modul_sasaran,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, pengaturan_lora)

model.print_trainable_parameters()

torch.cuda.empty_cache()

print(
    "Memori GPU terpakai:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GiB",
)
print("Persiapan selesai.")

trainable params: 557,056 || all params: 9,410,370,800 || trainable%: 0.0059
Memori GPU terpakai: 7.36 GiB
Persiapan selesai.


In [11]:
from datetime import datetime
from transformers import (
    Trainer,
    TrainingArguments,
    default_data_collator,
)

# Pastikan data latihan sudah tersedia.
if "dataset_latihan" not in globals():
    raise RuntimeError(
        "Jalankan kembali sel pembentukan dataset_latihan dahulu."
    )

# Buat folder baru untuk menyimpan percobaan ini.
folder_hasil = (
    "/content/drive/MyDrive/LeveLearn/latihan_dummy/"
    + datetime.now().strftime("%Y%m%d_%H%M%S")
)

pengaturan_latihan = TrainingArguments(
    output_dir=folder_hasil,

    # Satu contoh diproses setiap langkah.
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,

    # Percobaan singkat: 12 langkah pembaruan.
    max_steps=12,
    learning_rate=1e-4,

    # Pengaturan untuk menghemat memori.
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="adamw_torch",

    # Tampilkan perkembangan dan simpan setiap 6 langkah.
    logging_steps=1,
    save_strategy="steps",
    save_steps=6,
    save_total_limit=2,

    report_to="none",
    remove_unused_columns=False,
    seed=42,
)

trainer = Trainer(
    model=model,
    args=pengaturan_latihan,
    train_dataset=dataset_latihan,
    data_collator=default_data_collator,
)

print("Pelatihan dimulai.")
print("Folder hasil:", folder_hasil)

hasil_latihan = trainer.train()

# Simpan hasil akhir ke Google Drive.
model.save_pretrained(folder_hasil)
processor.save_pretrained(folder_hasil)

print("Pelatihan selesai dan hasil tersimpan.")
print("Lokasi:", folder_hasil)

Pelatihan dimulai.
Folder hasil: /content/drive/MyDrive/LeveLearn/latihan_dummy/20261101_125652


Step,Training Loss
1,1.985984
2,2.110140
3,2.076974
4,2.644508
5,2.223181
6,2.346902
7,2.644508
8,1.985984
9,2.340465
10,2.195350


Pelatihan selesai dan hasil tersimpan.
Lokasi: /content/drive/MyDrive/LeveLearn/latihan_dummy/20261101_125652


Sekarang kita periksa hasil pelatihannya. Jalankan dua sel berikut pada sesi yang sama; tidak perlu melatih ulang.
1. Periksa parameter tambahan
Pada pengaturan LoRA yang kita gunakan, bagian lora_B awalnya berisi nol. Kita periksa apakah setelah latihan ada nilai yang berubah.

In [12]:
import torch

bagian_b = [
    parameter.detach()
    for nama, parameter in model.named_parameters()
    if "lora_B" in nama
]

if not bagian_b:
    raise RuntimeError("Parameter LoRA tidak ditemukan.")

semua_finite = all(
    torch.isfinite(parameter).all().item()
    for parameter in bagian_b
)

nilai_terbesar = max(
    parameter.abs().max().item()
    for parameter in bagian_b
)

print("Semua nilai normal:", semua_finite)
print("Nilai absolut terbesar LoRA B:", nilai_terbesar)

if not semua_finite:
    print("Ada nilai tidak valid. Jangan lanjutkan pengujian dahulu.")
elif nilai_terbesar == 0:
    print("LoRA B masih nol. Pembaruan saat latihan perlu diperiksa.")
else:
    print("LoRA B sudah memiliki nilai bukan nol.")

Semua nilai normal: True
Nilai absolut terbesar LoRA B: 0.00041829171823337674
LoRA B sudah memiliki nilai bukan nol.


Jika nilainya bukan nol dan semuanya normal, lanjutkan. Ini menunjukkan perubahan parameter, belum membuktikan peningkatan kualitas.

2. Bandingkan jawaban tanpa dan dengan hasil latihan
Kita memakai pertanyaan yang tidak ada dalam enam contoh latihan. Saat LoRA dinonaktifkan sementara, model memakai bobot utama tanpa tambahan hasil latihan. Fasilitas ini tersedia di PEFT.

In [13]:
model.eval()
model.gradient_checkpointing_disable()

pertanyaan_baru = (
    "Jika p salah dan q benar, tentukan nilai p ∧ q. "
    "Jelaskan dalam bahasa Indonesia."
)

# Jawaban model tanpa tambahan hasil latihan.
with model.disable_adapter():
    jawaban_dasar = tanya_model(pertanyaan_baru)

# Setelah keluar dari blok tadi, LoRA kembali aktif.
jawaban_dilatih = tanya_model(pertanyaan_baru)

print("PERTANYAAN:")
print(pertanyaan_baru)

print("\nTANPA HASIL LATIHAN:")
print(jawaban_dasar)

print("\nDENGAN HASIL LATIHAN:")
print(jawaban_dilatih)

PERTANYAAN:
Jika p salah dan q benar, tentukan nilai p ∧ q. Jelaskan dalam bahasa Indonesia.

TANPA HASIL LATIHAN:
Dalam logika matematika, simbol **∧** (dibaca "dan") merepresentasikan operasi **konjungsi**. Nilai kebenaran dari konjungsi ($p \land q$) hanya bernilai **benar** jika kedua pernyataan penyusunnya ($p$ dan $q$) bernilai benar secara bersamaan. Jika salah satu atau keduanya salah, maka hasilnya adalah salah.

Diketahui dalam soal Anda:
*   $p$ bernilai **salah** (False).
*   $q$ bernilai **benar** (True).

Karena salah satu pernyataan ($p$) bernilai salah, maka hasil dari $p \land q$ akan menjadi salah, tanpa memandang nilai kebenaran dari $q$.

Jadi

DENGAN HASIL LATIHAN:
Dalam logika matematika, simbol **∧** (dibaca "dan") merepresentasikan operasi **konjungsi**. Nilai kebenaran dari konjungsi ($p \land q$) hanya bernilai **benar** jika kedua pernyataan penyusunnya ($p$ dan $q$) bernilai benar secara bersamaan. Jika salah satu atau keduanya salah, maka hasilnya adalah sa

In [14]:
import torch

model.eval()
model.gradient_checkpointing_disable()

def hitung_kesalahan():
    total_kesalahan = 0.0
    total_token_jawaban = 0

    with torch.no_grad():
        for contoh in dataset_latihan:
            masukan = {
                nama: torch.tensor(
                    [contoh[nama]],
                    dtype=torch.long,
                    device="cuda",
                )
                for nama in ["input_ids", "attention_mask", "labels"]
            }

            # Hitung jumlah token jawaban yang benar-benar dinilai.
            jumlah_token = (
                masukan["labels"][:, 1:] != -100
            ).sum().item()

            if jumlah_token == 0:
                continue

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
            ):
                hasil = model(
                    **masukan,
                    use_cache=False,
                )

            total_kesalahan += hasil.loss.item() * jumlah_token
            total_token_jawaban += jumlah_token

            del masukan, hasil

    if total_token_jawaban == 0:
        raise ValueError("Tidak ada token jawaban yang dapat dinilai.")

    return total_kesalahan / total_token_jawaban


# Model utama tanpa tambahan hasil latihan.
with model.disable_adapter():
    kesalahan_dasar = hitung_kesalahan()

# Model dengan tambahan hasil latihan.
kesalahan_dilatih = hitung_kesalahan()

print(f"Tanpa hasil latihan : {kesalahan_dasar:.6f}")
print(f"Dengan hasil latihan: {kesalahan_dilatih:.6f}")

selisih = kesalahan_dilatih - kesalahan_dasar
print(f"Selisih              : {selisih:+.6f}")

torch.cuda.empty_cache()

Tanpa hasil latihan : 2.218250
Dengan hasil latihan: 2.137466
Selisih              : -0.080784


In [15]:
pertanyaan_uji = [
    {
        "pertanyaan": (
            "Jika p salah dan q benar, tentukan nilai p ∧ q. "
            "Jelaskan dalam bahasa Indonesia."
        ),
        "kunci": "Salah",
    },
    {
        "pertanyaan": (
            "Jika p benar dan q salah, tentukan nilai p ∨ q. "
            "Jelaskan dalam bahasa Indonesia."
        ),
        "kunci": "Benar",
    },
    {
        "pertanyaan": (
            "Jika r benar, tentukan nilai ¬r. "
            "Jelaskan dalam bahasa Indonesia."
        ),
        "kunci": "Salah",
    },
]

model.eval()
model.gradient_checkpointing_disable()

hasil_uji = []

for nomor, contoh in enumerate(pertanyaan_uji, start=1):
    with model.disable_adapter():
        jawaban_dasar = tanya_model(contoh["pertanyaan"])

    jawaban_dilatih = tanya_model(contoh["pertanyaan"])

    hasil_uji.append({
        **contoh,
        "jawaban_dasar": jawaban_dasar,
        "jawaban_dilatih": jawaban_dilatih,
    })

    print(f"\n===== PERTANYAAN {nomor} =====")
    print(contoh["pertanyaan"])
    print("Kunci:", contoh["kunci"])

    print("\nTANPA HASIL LATIHAN:")
    print(jawaban_dasar)

    print("\nDENGAN HASIL LATIHAN:")
    print(jawaban_dilatih)


===== PERTANYAAN 1 =====
Jika p salah dan q benar, tentukan nilai p ∧ q. Jelaskan dalam bahasa Indonesia.
Kunci: Salah

TANPA HASIL LATIHAN:
Dalam logika matematika, simbol **∧** (dibaca "dan") merepresentasikan operasi **konjungsi**. Nilai kebenaran dari konjungsi ($p \land q$) hanya bernilai **benar** jika kedua pernyataan penyusunnya ($p$ dan $q$) bernilai benar secara bersamaan. Jika salah satu atau keduanya salah, maka hasilnya adalah salah.

Diketahui dalam soal Anda:
*   $p$ bernilai **salah** (False).
*   $q$ bernilai **benar** (True).

Karena salah satu pernyataan ($p$) bernilai salah, maka hasil dari $p \land q$ akan menjadi salah, tanpa memandang nilai kebenaran dari $q$.

Jadi

DENGAN HASIL LATIHAN:
Dalam logika matematika, simbol **∧** (dibaca "dan") merepresentasikan operasi **konjungsi**. Nilai kebenaran dari konjungsi ($p \land q$) hanya bernilai **benar** jika kedua pernyataan penyusunnya ($p$ dan $q$) bernilai benar secara bersamaan. Jika salah satu atau keduanya sal

***1 November***

1. Hubungkan akun Google

In [16]:
from pathlib import Path

nama_video = "Propositional Logic_ Binary Decision Diagram 1080P.mp4"
drive_root = Path("/content/drive/MyDrive")

if not drive_root.exists():
    raise RuntimeError("Google Drive belum ter-mount.")

daftar_video = list(drive_root.rglob(nama_video))

if not daftar_video:
    raise FileNotFoundError(
        "Video tidak ditemukan di MyDrive. Jika berada di folder "
        "'Dibagikan kepada saya', tambahkan pintasan folder ke MyDrive."
    )

if len(daftar_video) > 1:
    raise RuntimeError(
        "Ada beberapa file dengan nama sama. Pilih lokasi yang benar:\n"
        + "\n".join(str(p) for p in daftar_video)
    )

video_asli = daftar_video[0]
print("Video ditemukan:", video_asli)

KeyboardInterrupt: 